# Exposure analysis: data-transfer ON vs OFF

Loads the per-day and per-track tables produced by `multi_day_pipeline.ipynb` and asks: **does the data-transfer affect bumblebee foraging?**

## Notebook structure

The notebook is split into three conceptually different sections, deliberately not mixed together:

1. **Temporal context.** Time-series of daily activity, every date plotted in chronological order, with shading by condition. This is the only place where BASELINE and ON and OFF appear in the same figure - because the figure is about the *timeline*, not about contrasting conditions.
2. **Pre-exposure baseline characterisation.** What bees did before the transfer was ever turned on (dates before 2026-04-23). Establishes "normal" behaviour and seasonal context. No ON or OFF data appears here.
3. **Experimental contrast: ON vs OFF.** The actual comparison. BASELINE is excluded from these plots - it's a different kind of period (no equipment running yet, different week of the season) and mixing it into a boxplot or histogram invites misleading visual comparisons.

## Experimental design

Three-day-on, three-day-off cycle, anchored at **2026-04-23**. Anything before that is `BASELINE`.

| Period | Dates | Condition |
|---|---|---|
| Pre-exposure | up to 2026-04-22 | BASELINE |
| Cycle 1 ON  | 2026-04-23 to 2026-04-25 | ON |
| Cycle 1 OFF | 2026-04-26 to 2026-04-28 | OFF |
| Cycle 2 ON  | 2026-04-29 to 2026-05-01 | ON |
| Cycle 2 OFF | 2026-05-02 to 2026-05-04 | OFF |
| ... | (3-on / 3-off continues) | |

## Caveats up front

- **Sample size.** 3 days per condition per cycle. Treat large p-values as "we can't see an effect at this sample size", not "no effect exists".
- **Confounds.** Weather, seasonal forage availability, hive maturation. Pull KNMI Wageningen weather data and overlay if effects look suggestive.
- **One-camera-only effects.** A "treatment effect" that only shows on system 900 or 939 is more likely a camera-specific artefact than biology. Always check both cameras.
- **Trip duration is approximate.** PATS-C doesn't track bee identity across detections. We pair each v3 exit with the next un-matched return greedily, which means *individual* trip durations are noise (we can't say "this bee took 30 minutes") - but the *median across many trips* is robust to those swaps because the mismatches cancel symmetrically. So `median_trip_s` is fine as a comparison metric between conditions, just don't read it as a statement about any one bee.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


## 1. Treatment schedule

`condition_for(date)` -> `"BASELINE" | "ON" | "OFF"`. Edit the cycle anchor / length here if the schedule changes.


In [ ]:
CYCLE_ANCHOR   = pd.Timestamp("2026-04-23")
CYCLE_ON_DAYS  = 3
CYCLE_OFF_DAYS = 3
CYCLE_LEN      = CYCLE_ON_DAYS + CYCLE_OFF_DAYS

def condition_for(date_like):
    """date_like: anything pd.Timestamp can parse (str, Timestamp, datetime)."""
    d = pd.Timestamp(date_like).normalize()
    if d < CYCLE_ANCHOR:
        return "BASELINE"
    days_since = (d - CYCLE_ANCHOR).days
    cycle_pos  = days_since % CYCLE_LEN
    return "ON" if cycle_pos < CYCLE_ON_DAYS else "OFF"


print("Schedule preview:")
for d in pd.date_range("2026-04-19", "2026-05-08"):
    print(f"  {d.date()}  ->  {condition_for(d)}")


## 2. Load data


In [ ]:
DATA_DIR = Path("data/multi_day")
daily  = pd.read_csv(DATA_DIR / "daily_summary.csv")
tracks = pd.read_csv(DATA_DIR / "per_track_indicators.csv", parse_dates=["ts"])

daily["condition"]  = daily["date"].apply(condition_for)
tracks["condition"] = tracks["date"].apply(condition_for)

systems = sorted(daily["system_id"].unique())

print("daily summary by condition:")
print(daily.groupby("condition").agg(
    n_pairs    = ("date",        "size"),
    n_dates    = ("date",        "nunique"),
    total_v3   = ("n_v3",        "sum"),
    total_ret  = ("n_returns",   "sum"),
    median_trip= ("median_trip_s","median"),
).to_string())

print(f"\nper_track rows: {len(tracks):,}  ({tracks['condition'].value_counts().to_dict()})")


# Section A - Temporal context

Every date, in chronological order, with condition shading. This is the only figure that shows BASELINE / ON / OFF together: the point of this figure is the *timeline*, not the contrast.


## 3. Daily activity time series

Bar chart per camera. **Red shading** = transfer ON. **Grey shading** = pre-exposure baseline. White = OFF (transfer installed but turned off).


In [ ]:
fig, axes = plt.subplots(len(systems), 1, figsize=(13, 4 * len(systems)),
                          sharex=True, squeeze=False)

for ax, sys_id in zip(axes[:, 0], systems):
    sub = (daily[daily["system_id"] == sys_id]
           .sort_values("date").reset_index(drop=True))
    if sub.empty:
        ax.set_title(f"system {sys_id} - no data")
        continue
    x = pd.to_datetime(sub["date"])
    w = pd.Timedelta(hours=20)

    ax.bar(x, sub["n_v3"],      width=w,            color="tab:blue",   alpha=0.85, label="v3 exits")
    ax.bar(x, sub["n_returns"], width=w * 0.55,     color="tab:orange", alpha=0.95, label="returns")

    for _, row in sub.iterrows():
        d = pd.Timestamp(row["date"])
        if row["condition"] == "ON":
            ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12),
                       color="red", alpha=0.10, zorder=0)
        elif row["condition"] == "BASELINE":
            ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12),
                       color="grey", alpha=0.10, zorder=0)

    ax.set_title(f"system {sys_id}")
    ax.set_ylabel("count")
    ax.legend(loc="upper right")
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Daily exits & returns  -  red = ON, grey = BASELINE, white = OFF", y=1.02)
plt.tight_layout()
plt.show()


# Section B - Pre-exposure baseline characterisation

What did the bees do *before* anything was turned on? This section establishes the "normal" picture against which any later change should be read. **No ON or OFF data appears here.**

If you don't have any pre-23 dates cached, this section will print "no baseline data" and skip.


In [ ]:
baseline_daily  = daily[daily["condition"] == "BASELINE"].copy()
baseline_tracks = tracks[tracks["condition"] == "BASELINE"].copy()

if baseline_daily.empty:
    print("No baseline data in daily_summary. Skipping section B.")
else:
    print(f"Baseline window: {baseline_daily['date'].min()}  to  {baseline_daily['date'].max()}")
    print(f"Baseline days: {baseline_daily['date'].nunique()}")
    print(f"Baseline (date, system) pairs: {len(baseline_daily)}")
    print(f"Baseline tracks: {len(baseline_tracks):,}")
    print()
    print("Per-system summary:")
    print(baseline_daily.groupby("system_id").agg(
        n_dates       = ("date",         "nunique"),
        median_v3     = ("n_v3",         "median"),
        median_ret    = ("n_returns",    "median"),
        median_trip_s = ("median_trip_s", "median"),
    ).to_string())


### B.1 Baseline daily counts (one box per camera)

Spread of `n_v3` and `n_returns` across baseline days. Tells you the **natural** day-to-day variance the bees show, before any treatment. If the experimental contrast (Section C below) reports a difference smaller than this baseline spread, it's not a real effect - just noise within natural variance.


In [ ]:
if baseline_daily.empty:
    print("No baseline data - skip.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, lbl in zip(axes, ["n_v3", "n_returns"], ["v3 exits / day", "returns / day"]):
        data = [baseline_daily.loc[baseline_daily["system_id"] == s, col].dropna()
                for s in systems]
        bp = ax.boxplot(data, tick_labels=[f"sys {s}" for s in systems],
                        patch_artist=True, widths=0.6)
        for patch in bp["boxes"]:
            patch.set_facecolor("lightgrey"); patch.set_alpha(0.85)
        for i, d in enumerate(data):
            ax.scatter(np.full(len(d), i + 1) + np.random.uniform(-0.06, 0.06, len(d)),
                       d, color="black", s=20, alpha=0.7, zorder=3)
        ax.set_ylabel(lbl)
        ax.set_title(f"BASELINE - {lbl}")
    plt.tight_layout()
    plt.show()


### B.2 Baseline path tortuosity

Distribution of per-track tortuosity during baseline. Reference shape for the ON-vs-OFF comparison below.


In [ ]:
if baseline_tracks.empty:
    print("No baseline tracks - skip.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    for sys_id, color in zip(systems, ["tab:purple", "tab:cyan"]):
        vals = (baseline_tracks
                .loc[(baseline_tracks["system_id"] == sys_id) &
                     baseline_tracks["tortuosity"].notna(), "tortuosity"]
                .clip(upper=10))
        if len(vals) == 0:
            continue
        ax.hist(vals, bins=50, alpha=0.55, color=color, density=True,
                label=f"sys {sys_id}  (n={len(vals)})", edgecolor="black", linewidth=0.2)
    ax.set_xlabel("tortuosity (arc / displacement, clipped at 10)")
    ax.set_ylabel("density")
    ax.set_title("BASELINE tortuosity distribution per camera")
    ax.legend()
    plt.tight_layout()
    plt.show()


### B.3 Baseline hourly activity curve

Average exits-per-hour curve across baseline days. Tells you when bees are normally active.


In [ ]:
def hourly_mean_curve(df, system_id, condition):
    sub = df[(df["system_id"] == system_id) &
             (df["condition"] == condition) &
             (df["hive_exit_v3"] == True) &
             df["ts"].notna()].copy()
    if sub.empty:
        return None
    sub["hour"] = sub["ts"].dt.hour
    sub["date"] = sub["ts"].dt.date
    counts = (sub.groupby(["date", "hour"]).size()
              .unstack(fill_value=0)
              .reindex(columns=range(24), fill_value=0))
    return counts.mean(axis=0)


if baseline_tracks.empty:
    print("No baseline tracks - skip.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    for sys_id, color in zip(systems, ["tab:purple", "tab:cyan"]):
        curve = hourly_mean_curve(tracks, sys_id, "BASELINE")
        if curve is None:
            continue
        ax.plot(curve.index, curve.values, "o-", color=color, lw=2,
                label=f"sys {sys_id}")
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("hour of day (Europe/Amsterdam)")
    ax.set_ylabel("mean v3 exits / hour")
    ax.set_title("BASELINE - average hourly exit rate")
    ax.legend()
    plt.tight_layout()
    plt.show()


# Section C - Experimental contrast: ON vs OFF

The actual experimental comparison. **BASELINE is excluded from this section** - we're answering "given the equipment is installed, does turning the transfer on change foraging?", not "do bees behave differently in late April than early May?". Mixing the two questions in one figure is what we want to avoid.


## 4. ON vs OFF box plots


In [ ]:
METRICS = [("n_v3",          "v3 exits / day"),
           ("n_returns",     "returns / day"),
           ("median_trip_s", "median trip [s]")]
ORDER   = ["OFF", "ON"]
COLORS  = {"OFF": "tab:green", "ON": "tab:red"}

contrast_daily = daily[daily["condition"].isin(ORDER)].copy()

if contrast_daily.empty:
    print("No ON/OFF data yet. Skipping section C.")
else:
    fig, axes = plt.subplots(len(METRICS), len(systems),
                              figsize=(5 * len(systems), 3.5 * len(METRICS)),
                              squeeze=False)
    for row, (col, label) in enumerate(METRICS):
        for c_ax, sys_id in enumerate(systems):
            ax = axes[row, c_ax]
            sub = contrast_daily[contrast_daily["system_id"] == sys_id]
            data = [sub.loc[sub["condition"] == cond, col].dropna() for cond in ORDER]
            bp = ax.boxplot(data, tick_labels=ORDER, patch_artist=True, widths=0.55)
            for patch, cond in zip(bp["boxes"], ORDER):
                patch.set_facecolor(COLORS[cond]); patch.set_alpha(0.7)
            for i, d in enumerate(data):
                ax.scatter(np.full(len(d), i + 1) + np.random.uniform(-0.06, 0.06, len(d)),
                           d, color="black", s=18, alpha=0.7, zorder=3)
            ax.set_title(f"sys {sys_id} - {label}")
            ax.set_ylabel(label)
    fig.suptitle("ON vs OFF - daily metrics (each black dot is one day)", y=1.01)
    plt.tight_layout()
    plt.show()


## 5. ON vs OFF tortuosity

Per-track tortuosity distributions. If bees navigate less efficiently when the transfer is on, the ON histogram should sit slightly to the right of OFF.


In [ ]:
contrast_tracks = tracks[tracks["condition"].isin(["ON", "OFF"])].copy()

if contrast_tracks.empty:
    print("No ON/OFF tracks yet. Skipping.")
else:
    fig, axes = plt.subplots(1, len(systems), figsize=(7 * len(systems), 4),
                              sharey=True, squeeze=False)
    for ax, sys_id in zip(axes[0], systems):
        sub = contrast_tracks[(contrast_tracks["system_id"] == sys_id) &
                              contrast_tracks["tortuosity"].notna()]
        for cond, color in [("OFF", "tab:green"), ("ON", "tab:red")]:
            vals = sub.loc[sub["condition"] == cond, "tortuosity"].clip(upper=10)
            if len(vals) == 0:
                continue
            ax.hist(vals, bins=50, alpha=0.45, color=color, density=True,
                    label=f"{cond} (n={len(vals)})", edgecolor="black", linewidth=0.2)
        ax.set_xlabel("tortuosity (clipped at 10)")
        ax.set_ylabel("density")
        ax.set_title(f"system {sys_id}")
        ax.legend()
    fig.suptitle("ON vs OFF - path tortuosity per camera", y=1.02)
    plt.tight_layout()
    plt.show()


## 6. ON vs OFF hourly curve

Average hourly exit rate, ON vs OFF, with BASELINE plotted as a faint dashed line for *visual reference only* - not as a third condition in the comparison.


In [ ]:
if contrast_tracks.empty:
    print("No ON/OFF tracks yet. Skipping.")
else:
    fig, axes = plt.subplots(1, len(systems), figsize=(7 * len(systems), 4),
                              sharey=True, squeeze=False)
    for ax, sys_id in zip(axes[0], systems):
        for cond, color, lw, ls, alpha in [
            ("BASELINE", "grey",      1.5, "--", 0.55),  # reference only
            ("OFF",      "tab:green", 2.0, "-",  1.0),
            ("ON",       "tab:red",   2.0, "-",  1.0),
        ]:
            curve = hourly_mean_curve(tracks, sys_id, cond)
            if curve is None:
                continue
            ax.plot(curve.index, curve.values, "o" if ls == "-" else "",
                    color=color, lw=lw, ls=ls, alpha=alpha,
                    label=cond if ls == "-" else f"{cond} (reference)")
        ax.set_xticks(range(0, 24, 2))
        ax.set_xlabel("hour of day (Europe/Amsterdam)")
        ax.set_ylabel("mean v3 exits / hour")
        ax.set_title(f"system {sys_id}")
        ax.legend()
    fig.suptitle("ON vs OFF hourly exits  (baseline shown dashed for reference)", y=1.02)
    plt.tight_layout()
    plt.show()


## 7. Statistical tests (ON vs OFF)

Two-sided Mann-Whitney U for each metric, per camera. Non-parametric because daily counts don't have to be Gaussian and we have small samples. **Compares ON vs OFF only - BASELINE is not included** because it's a different kind of period (different week of the season, no transmitter installed).

**How to read the p-values.** With 3-6 days per condition, anything p < 0.05 is plausible but not bulletproof - it's worth a closer look but not a publishable claim on its own. p > 0.20 means "we can't distinguish ON from OFF at this sample size". Add more cycles before drawing strong conclusions.


In [ ]:
from scipy import stats

print(f"{'metric':25s} {'system':>6s} {'ON med':>10s} {'OFF med':>10s} {'p (MWU)':>10s} {'n_on':>5s} {'n_off':>5s}")
print("-" * 78)
for sys_id in systems:
    sub = daily[(daily["system_id"] == sys_id) & daily["condition"].isin(["ON", "OFF"])]
    for col, label in [("n_v3",          "n_v3"),
                       ("n_returns",     "n_returns"),
                       ("median_trip_s", "median_trip_s")]:
        on  = sub.loc[sub["condition"] == "ON",  col].dropna()
        off = sub.loc[sub["condition"] == "OFF", col].dropna()
        if len(on) == 0 or len(off) == 0:
            print(f"{label:25s} {sys_id:6d} {'-':>10s} {'-':>10s} {'n/a':>10s} {len(on):>5d} {len(off):>5d}")
            continue
        u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
        print(f"{label:25s} {sys_id:6d} {on.median():10.2f} {off.median():10.2f} {p:10.3f} {len(on):>5d} {len(off):>5d}")

print()
for sys_id in systems:
    sub = tracks[(tracks["system_id"] == sys_id) &
                 tracks["condition"].isin(["ON", "OFF"]) &
                 tracks["tortuosity"].notna()]
    on  = sub.loc[sub["condition"] == "ON",  "tortuosity"]
    off = sub.loc[sub["condition"] == "OFF", "tortuosity"]
    if len(on) == 0 or len(off) == 0:
        print(f"{'tortuosity (per track)':25s} {sys_id:6d} insufficient data")
        continue
    u, p = stats.mannwhitneyu(on, off, alternative="two-sided")
    print(f"{'tortuosity (per track)':25s} {sys_id:6d} {on.median():10.2f} {off.median():10.2f} {p:10.3f} {len(on):>5d} {len(off):>5d}")


## 8. Putting it together

- **Baseline (Section B) tells you the natural variance** - the spread you should expect between any two random days even with nothing changing. Any ON-vs-OFF effect smaller than this baseline spread is not credible.
- **Section C tells you whether ON and OFF differ** beyond what would be expected from natural day-to-day variation.
- A real treatment effect should ideally show: (1) consistent direction across both cameras, (2) larger than the baseline spread, (3) reproducible across cycles - i.e. cycle 1 ON vs cycle 1 OFF gives the same direction as cycle 2 ON vs cycle 2 OFF, not just an averaged difference that's actually a one-time blip.

When you add new cached days, just re-run `multi_day_pipeline.ipynb`, then re-run this notebook top-to-bottom. The condition tags update automatically.
